# 02 — Preprocesamiento de señales

Aplicamos normalización, remoción de silencios y filtrado básico antes del análisis espectral.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from scipy.signal import butter, filtfilt
from pathlib import Path
import IPython.display as ipd

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data')
ruta = sorted(DATA_DIR.glob('*.wav'))[0]
y, sr = librosa.load(ruta, sr=None)
print(f'Audio cargado: {ruta.name} | sr={sr} Hz | dur={len(y)/sr:.2f}s')

## 1. Normalización de amplitud

In [ ]:
y_norm = y / np.max(np.abs(y))

fig, axes = plt.subplots(2, 1, sharex=True)
t = np.linspace(0, len(y)/sr, len(y))
axes[0].plot(t, y, linewidth=0.5, color='gray')
axes[0].set_title('Original')
axes[0].set_ylabel('Amplitud')
axes[1].plot(t, y_norm, linewidth=0.5, color='steelblue')
axes[1].set_title('Normalizada (pico)')
axes[1].set_ylabel('Amplitud')
axes[1].set_xlabel('Tiempo [s]')
plt.tight_layout()
plt.savefig('../figuras/02_normalizacion.png', dpi=150)
plt.show()

## 2. Remoción de silencios (Voice Activity Detection simple)

In [ ]:
# Detectar intervalos no silenciosos
intervalos = librosa.effects.split(y_norm, top_db=30)
print(f'Segmentos de voz encontrados: {len(intervalos)}')

# Concatenar solo las partes con voz
y_voz = np.concatenate([y_norm[i:j] for i, j in intervalos])
print(f'Duración original : {len(y_norm)/sr:.2f} s')
print(f'Duración con voz  : {len(y_voz)/sr:.2f} s')

In [ ]:
fig, ax = plt.subplots()
t = np.linspace(0, len(y_norm)/sr, len(y_norm))
ax.plot(t, y_norm, linewidth=0.4, color='lightgray', label='señal completa')
for i, (inicio, fin) in enumerate(intervalos):
    ax.axvspan(inicio/sr, fin/sr, alpha=0.3, color='steelblue', label='voz' if i == 0 else None)
ax.set_xlabel('Tiempo [s]')
ax.set_ylabel('Amplitud')
ax.set_title('Detección de actividad vocal')
ax.legend()
plt.tight_layout()
plt.savefig('../figuras/02_vad.png', dpi=150)
plt.show()

## 3. Filtro pasa-banda (300 Hz – 3400 Hz)

Rango típico de la voz humana en telefonía.

In [ ]:
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

b, a = butter_bandpass(300, 3400, sr)
y_filtrada = filtfilt(b, a, y_voz)

fig, axes = plt.subplots(2, 1, sharex=True)
t = np.linspace(0, len(y_voz)/sr, len(y_voz))
axes[0].plot(t, y_voz, linewidth=0.5, color='gray')
axes[0].set_title('Señal de voz (sin filtrar)')
axes[0].set_ylabel('Amplitud')
axes[1].plot(t, y_filtrada, linewidth=0.5, color='steelblue')
axes[1].set_title('Señal filtrada (300 – 3400 Hz)')
axes[1].set_ylabel('Amplitud')
axes[1].set_xlabel('Tiempo [s]')
plt.tight_layout()
plt.savefig('../figuras/02_filtrado.png', dpi=150)
plt.show()

## 4. Guardar señal preprocesada

In [ ]:
sf.write(DATA_DIR / 'procesado.wav', y_filtrada, sr)
print('Señal preprocesada guardada en data/procesado.wav')
ipd.Audio(y_filtrada, rate=sr)